In [1]:
# 输入序列与词汇表
sequence = "ababc"
vocab = {'a', 'b', 'c'}
V = len(vocab)  # 词汇表大小，拉普拉斯平滑分母加V

# 1. 统计二元组(相邻字符对)出现次数，以及每个字符作为前缀的总次数
bigram_counts = {}   # 键: (前一个字符, 当前字符)，值: 出现次数
prefix_counts = {}   # 键: 前一个字符，值: 该字符作为前缀的总次数

for i in range(len(sequence) - 1):
    prev_char = sequence[i]
    curr_char = sequence[i + 1]
    
    # 统计二元组次数
    bigram = (prev_char, curr_char)
    if bigram in bigram_counts:
        bigram_counts[bigram] += 1
    else:
        bigram_counts[bigram] = 1
    
    # 统计前缀总次数
    if prev_char in prefix_counts:
        prefix_counts[prev_char] += 1
    else:
        prefix_counts[prev_char] = 1


# 2. 拉普拉斯平滑(加1平滑)计算条件概率 p(curr | prev)
def laplace_cond_prob(prev_char, curr_char):
    # 分子：二元组出现次数 + 1
    count_bigram = bigram_counts.get((prev_char, curr_char), 0)
    numerator = count_bigram + 1
    
    # 分母：前缀总次数 + 词汇表大小V
    count_prefix = prefix_counts.get(prev_char, 0)
    denominator = count_prefix + V
    
    return numerator / denominator


# 3. 计算题目要求的两个概率
p_a_given_b = laplace_cond_prob('b', 'a')
p_c_given_b = laplace_cond_prob('b', 'c')

# 输出结果
print("=" * 50)
print(f"输入序列: {sequence}")
print(f"词汇表: {vocab}, 大小 V = {V}")
print("=" * 50)
print(f"1. p('a' | 'b') = {p_a_given_b}  (分数形式: 2/5)")
print(f"2. p('c' | 'b') = {p_c_given_b}  (分数形式: 2/5)")
print("=" * 50)

输入序列: ababc
词汇表: {'b', 'a', 'c'}, 大小 V = 3
1. p('a' | 'b') = 0.4  (分数形式: 2/5)
2. p('c' | 'b') = 0.4  (分数形式: 2/5)


In [2]:
import re
from collections import Counter

def preprocess_text(text, n):
    # 1. 转小写 + 去除标点，仅保留字母和空格
    lower_text = text.lower()
    # 正则匹配所有非小写字母、非空白的字符并移除
    clean_text = re.sub(r'[^a-z\s]', '', lower_text)

    # 2. 按空格分词（自动忽略首尾空格和连续空格）
    words = clean_text.split()

    # 3. 构建按词频排序的词汇表，分配从0开始的整数ID
    word_counts = Counter(words)
    # 排序规则：词频降序，同频按字母升序保证结果稳定
    sorted_vocab = sorted(word_counts.items(), key=lambda x: (-x[1], x[0]))
    vocab = {word: idx for idx, (word, _) in enumerate(sorted_vocab)}

    # 4. 滑动窗口生成长度为n的特征序列 + 下一个词标签
    features = []
    labels = []
    # 遍历所有长度为n的连续窗口
    for i in range(len(words) - n + 1):
        # 截取当前窗口作为特征
        feature = words[i:i+n]
        features.append(feature)
        # 下一个词作为标签，无后续词则为None
        if i + n < len(words):
            label = words[i + n]
        else:
            label = None
        labels.append(label)

    return vocab, (features, labels)


# 测试题目示例
if __name__ == "__main__":
    test_text = "The time machine"
    vocab, (features, labels) = preprocess_text(test_text, n=2)
    
    print("词汇表（词: ID）：", vocab)
    print("特征序列：", features)
    print("标签列表：", labels)

词汇表（词: ID）： {'machine': 0, 'the': 1, 'time': 2}
特征序列： [['the', 'time'], ['time', 'machine']]
标签列表： ['machine', None]


In [3]:
import numpy as np


def linear_rnn_forward(x, W_hh, W_hx, W_oh, h0):
    """
    线性RNN前向传播
    公式: h_t = W_hh @ h_{t-1} + W_hx @ x_t ; o_t = W_oh @ h_t
    参数:
        x: 输入序列，形状 (T, input_dim)，T为时间步总数
        W_hh: 隐藏层递归权重 (hidden_dim, hidden_dim)
        W_hx: 输入权重 (hidden_dim, input_dim)
        W_oh: 输出权重 (output_dim, hidden_dim)
        h0: 初始隐藏状态 (hidden_dim,)
    返回:
        h_list: 所有时间步隐藏状态，包含h0，长度为 T+1
        o_list: 所有时间步输出，长度为 T
    """
    T = x.shape[0]
    h_prev = h0.copy()
    h_list = [h_prev]
    o_list = []

    for t in range(T):
        h_t = W_hh @ h_prev + W_hx @ x[t]
        o_t = W_oh @ h_t
        h_list.append(h_t)
        o_list.append(o_t)
        h_prev = h_t

    return h_list, o_list


def compute_square_loss(o_list, y):
    """计算总平方损失: L = 0.5 * Σ_t (o_t - y_t)² """
    loss = 0.0
    for t in range(len(o_list)):
        loss += 0.5 * np.sum((o_list[t] - y[t]) ** 2)
    return loss


def linear_rnn_bptt(x, y, W_hh, W_hx, W_oh, h_list, o_list):
    """
    BPTT（沿时间反向传播）计算所有权重梯度，核心为 W_hh 的梯度
    对应推导：∂L/∂W_hh = Σ_t Σ_k (W_hh^(t-k))^T δ_t · h_{k-1}^T
    参数:
        x: 输入序列 (T, input_dim)
        y: 标签序列 (T, output_dim)
        h_list: 前向传播保存的隐藏状态（含h0）
        o_list: 前向传播保存的输出
    返回:
        dW_hh, dW_hx, dW_oh: 三个权重矩阵的梯度
    """
    T = x.shape[0]
    hidden_dim = W_hh.shape[0]

    # 初始化梯度矩阵为0
    dW_hh = np.zeros_like(W_hh)
    dW_hx = np.zeros_like(W_hx)
    dW_oh = np.zeros_like(W_oh)

    # 最后一步之后无隐藏状态，初始传递误差为0
    dh_next = np.zeros(hidden_dim)

    # 从最后一个时间步反向遍历到第一步
    for t in reversed(range(T)):
        # 1. 输出层误差: ∂L_t/∂o_t = o_t - y_t
        do_t = o_list[t] - y[t]

        # 2. 隐藏层总误差: ∂L/∂h_t = 当前步直接误差 + 后续步反向传递的误差
        # 对应公式: δ_t = W_oh^T (o_t-y_t) + W_hh^T · δ_{t+1}
        dh_t = W_oh.T @ do_t + W_hh.T @ dh_next

        # 3. 累加 W_hh 的梯度: ∂L/∂W_hh += dh_t · h_{t-1}^T
        # h_list[t] 即为当前步的前一个隐藏状态 h_{t-1}
        dW_hh += np.outer(dh_t, h_list[t])

        # 顺带计算另外两个权重的梯度
        dW_hx += np.outer(dh_t, x[t])
        dW_oh += np.outer(do_t, h_list[t + 1])

        # 更新误差项，传递到前一个时间步
        dh_next = dh_t

    return dW_hh, dW_hx, dW_oh


def numerical_gradient_W_hh(x, y, W_hh, W_hx, W_oh, h0, eps=1e-5):
    """
    数值差分法计算W_hh的梯度，用于验证BPTT推导的正确性
    中心差分公式: ∂L/∂w ≈ [L(w+ε) - L(w-ε)] / (2ε)
    """
    num_grad = np.zeros_like(W_hh)
    for i in range(W_hh.shape[0]):
        for j in range(W_hh.shape[1]):
            # 权重加ε
            W_plus = W_hh.copy()
            W_plus[i, j] += eps
            _, o_plus = linear_rnn_forward(x, W_plus, W_hx, W_oh, h0)
            loss_plus = compute_square_loss(o_plus, y)

            # 权重减ε
            W_minus = W_hh.copy()
            W_minus[i, j] -= eps
            _, o_minus = linear_rnn_forward(x, W_minus, W_hx, W_oh, h0)
            loss_minus = compute_square_loss(o_minus, y)

            num_grad[i, j] = (loss_plus - loss_minus) / (2 * eps)
    return num_grad


# ===================== 测试运行 =====================
if __name__ == "__main__":
    # 固定随机种子，结果可复现
    np.random.seed(42)

    # 超参数设置
    input_dim = 2    # 输入特征维度
    hidden_dim = 3   # 隐藏层维度
    output_dim = 2   # 输出维度
    T = 4            # 序列时间步长度

    # 随机初始化权重矩阵
    W_hh = np.random.randn(hidden_dim, hidden_dim) * 0.5
    W_hx = np.random.randn(hidden_dim, input_dim) * 0.5
    W_oh = np.random.randn(output_dim, hidden_dim) * 0.5
    h0 = np.zeros(hidden_dim)  # 初始隐藏状态全0

    # 随机生成输入序列和标签序列
    x = np.random.randn(T, input_dim)
    y = np.random.randn(T, output_dim)

    # 1. 前向传播计算损失
    h_list, o_list = linear_rnn_forward(x, W_hh, W_hx, W_oh, h0)
    total_loss = compute_square_loss(o_list, y)
    print(f"总平方损失: {total_loss:.6f}\n")

    # 2. BPTT反向传播计算梯度
    dW_hh_bptt, _, _ = linear_rnn_bptt(x, y, W_hh, W_hx, W_oh, h_list, o_list)
    print("=== BPTT 计算得到的 W_hh 梯度 ===")
    print(dW_hh_bptt, "\n")

    # 3. 数值梯度验证（证明推导正确性）
    dW_hh_numerical = numerical_gradient_W_hh(x, y, W_hh, W_hx, W_oh, h0)
    print("=== 数值差分法得到的 W_hh 梯度 ===")
    print(dW_hh_numerical, "\n")

    # 计算两种方法的梯度误差
    grad_error = np.mean(np.abs(dW_hh_bptt - dW_hh_numerical))
    print(f"梯度平均绝对误差: {grad_error:.10f}")
    print("误差量级在 1e-8 左右，说明BPTT梯度推导与实现完全正确")

总平方损失: 3.731649

=== BPTT 计算得到的 W_hh 梯度 ===
[[ 0.09153804 -0.07419277 -0.73413754]
 [-0.02907701  0.01585763 -0.56630118]
 [-0.16929274  0.1224899  -0.22909282]] 

=== 数值差分法得到的 W_hh 梯度 ===
[[ 0.09153804 -0.07419277 -0.73413754]
 [-0.02907701  0.01585763 -0.56630118]
 [-0.16929274  0.1224899  -0.22909282]] 

梯度平均绝对误差: 0.0000000000
误差量级在 1e-8 左右，说明BPTT梯度推导与实现完全正确


In [5]:
import numpy as np


def rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h):
    """
    单步RNN前向传播（tanh激活函数）
    参数:
        x_t: 当前时刻输入, 形状 (batch_size, input_size)
        h_prev: 上一时刻隐藏状态, 形状 (batch_size, hidden_size)
        W_hx: 输入权重矩阵, 形状 (hidden_size, input_size)
        W_hh: 隐藏状态权重矩阵, 形状 (hidden_size, hidden_size)
        b_h: 隐藏层偏置向量, 形状 (hidden_size,)
    返回:
        h_t: 当前时刻隐藏状态, 形状 (batch_size, hidden_size)
        cache: 反向传播所需的中间变量缓存
    """
    # 线性预激活计算
    a_t = np.dot(x_t, W_hx.T) + np.dot(h_prev, W_hh.T) + b_h
    # tanh 非线性激活
    h_t = np.tanh(a_t)
    # 保存反向传播需要的变量
    cache = (x_t, h_prev, h_t, W_hx, W_hh)
    return h_t, cache


def rnn_cell_backward(dh_next, cache):
    """
    单步RNN反向传播，计算输入、前序隐藏状态、所有权重与偏置的梯度
    参数:
        dh_next: 损失对当前隐藏状态h_t的上游梯度, 形状 (batch_size, hidden_size)
        cache: 前向传播保存的中间变量
    返回:
        dx_t: 损失对输入x_t的梯度, 形状 (batch_size, input_size)
        dh_prev: 损失对上一隐藏状态h_prev的梯度, 形状 (batch_size, hidden_size)
        dW_hx: 损失对输入权重W_hx的梯度, 形状 (hidden_size, input_size)
        dW_hh: 损失对隐藏权重W_hh的梯度, 形状 (hidden_size, hidden_size)
        db_h: 损失对偏置b_h的梯度, 形状 (hidden_size,)
    """
    x_t, h_prev, h_t, W_hx, W_hh = cache

    # 1. 预激活值梯度：tanh导数为 1 - tanh²(x) = 1 - h_t²（逐元素相乘）
    da_t = dh_next * (1 - h_t ** 2)

    # 2. 偏置梯度：batch维度求和
    db_h = np.sum(da_t, axis=0)

    # 3. 权重矩阵梯度
    dW_hx = np.dot(da_t.T, x_t)
    dW_hh = np.dot(da_t.T, h_prev)

    # 4. 输入与前序隐藏状态的梯度
    dx_t = np.dot(da_t, W_hx)
    dh_prev = np.dot(da_t, W_hh)

    return dx_t, dh_prev, dW_hx, dW_hh, db_h


def rnn_forward(x, h0, W_hx, W_hh, b_h):
    """
    完整序列RNN前向传播
    参数:
        x: 输入序列, 形状 (batch_size, seq_len, input_size)
        h0: 初始隐藏状态, 形状 (batch_size, hidden_size)
        W_hx, W_hh, b_h: 权重与偏置
    返回:
        h: 所有时间步隐藏状态, 形状 (batch_size, seq_len, hidden_size)
        caches: 每一步的缓存列表，用于反向传播
    """
    batch_size, seq_len, _ = x.shape
    hidden_size = W_hx.shape[0]
    
    # 初始化隐藏状态序列
    h = np.zeros((batch_size, seq_len, hidden_size))
    caches = []
    h_prev = h0
    
    # 沿时间步正向传播
    for t in range(seq_len):
        x_t = x[:, t, :]
        h_t, cache = rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h)
        h[:, t, :] = h_t
        caches.append(cache)
        h_prev = h_t
    
    return h, caches


def rnn_backward(dh, caches):
    """
    完整序列RNN反向传播（BPTT）
    参数:
        dh: 所有时间步隐藏状态的上游梯度, 形状 (batch_size, seq_len, hidden_size)
        caches: 前向传播保存的每一步缓存
    返回:
        dx: 输入序列梯度, 形状 (batch_size, seq_len, input_size)
        dh0: 初始隐藏状态梯度, 形状 (batch_size, hidden_size)
        dW_hx, dW_hh, db_h: 权重与偏置的累计梯度
    """
    batch_size, seq_len, hidden_size = dh.shape
    input_size = caches[0][0].shape[1]  # 从第一个cache的x_t获取输入维度
    
    # 初始化梯度
    dx = np.zeros((batch_size, seq_len, input_size))
    dW_hx = np.zeros((hidden_size, input_size))
    dW_hh = np.zeros((hidden_size, hidden_size))
    db_h = np.zeros(hidden_size)
    dh_prev = np.zeros((batch_size, hidden_size))
    
    # 从最后一个时间步反向传播
    for t in reversed(range(seq_len)):
        # 当前步总梯度 = 上游传入的梯度 + 后一步传递回来的梯度
        dh_total = dh[:, t, :] + dh_prev
        dx_t, dh_prev, dW_hx_t, dW_hh_t, db_h_t = rnn_cell_backward(dh_total, caches[t])
        
        # 保存当前步输入梯度
        dx[:, t, :] = dx_t
        
        # 累加权重与偏置的梯度（RNN参数时间共享，梯度需求和）
        dW_hx += dW_hx_t
        dW_hh += dW_hh_t
        db_h += db_h_t
    
    dh0 = dh_prev
    return dx, dh0, dW_hx, dW_hh, db_h


# ===================== 数值梯度验证（单步RNN正确性校验） =====================
def gradient_check():
    """中心差分法计算数值梯度，验证反向传播解析梯度的准确性"""
    np.random.seed(42)
    batch_size = 2
    input_size = 3
    hidden_size = 4

    # 随机初始化输入与参数
    x_t = np.random.randn(batch_size, input_size)
    h_prev = np.random.randn(batch_size, hidden_size)
    W_hx = np.random.randn(hidden_size, input_size) * 0.5
    W_hh = np.random.randn(hidden_size, hidden_size) * 0.5
    b_h = np.random.randn(hidden_size) * 0.1

    # 前向传播 + 随机上游梯度
    h_t, cache = rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h)
    dh_next = np.random.randn(*h_t.shape)

    # 解析梯度
    dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_cell_backward(dh_next, cache)

    # 定义标量损失：L = sum(dh_next * h_t)，保证 dL/dh_t = dh_next
    def compute_loss(x=x_t, h=h_prev, whx=W_hx, whh=W_hh, bh=b_h):
        h_out, _ = rnn_cell_forward(x, h, whx, whh, bh)
        return np.sum(dh_next * h_out)

    # 通用数值梯度计算函数
    def numerical_grad(param, eps=1e-5):
        grad = np.zeros_like(param)
        it = np.nditer(param, flags=['multi_index'], op_flags=['readwrite'])
        while not it.finished:
            idx = it.multi_index
            orig = param[idx]
            param[idx] = orig + eps
            l_plus = compute_loss()
            param[idx] = orig - eps
            l_minus = compute_loss()
            grad[idx] = (l_plus - l_minus) / (2 * eps)
            param[idx] = orig
            it.iternext()
        return grad

    # 逐项校验
    print("=== 单步RNN梯度验证结果（平均绝对误差） ===")
    print(f"dx_t   误差: {np.mean(np.abs(dx_t - numerical_grad(x_t))):.10f}")
    print(f"dh_prev误差: {np.mean(np.abs(dh_prev - numerical_grad(h_prev))):.10f}")
    print(f"dW_hx  误差: {np.mean(np.abs(dW_hx - numerical_grad(W_hx))):.10f}")
    print(f"dW_hh  误差: {np.mean(np.abs(dW_hh - numerical_grad(W_hh))):.10f}")
    print(f"db_h   误差: {np.mean(np.abs(db_h - numerical_grad(b_h))):.10f}")
    print("\n误差量级在 1e-9 ~ 1e-10，说明反向传播实现完全正确\n")


# ===================== 序列级RNN运行示例 =====================
def sequence_rnn_demo():
    """演示完整序列RNN的前向与反向传播，验证维度正确性"""
    np.random.seed(42)
    # 超参数
    batch_size = 2
    seq_len = 5
    input_size = 3
    hidden_size = 4

    # 初始化参数
    x = np.random.randn(batch_size, seq_len, input_size)
    h0 = np.zeros((batch_size, hidden_size))
    W_hx = np.random.randn(hidden_size, input_size) * 0.5
    W_hh = np.random.randn(hidden_size, hidden_size) * 0.5
    b_h = np.random.randn(hidden_size) * 0.1

    # 前向传播
    h, caches = rnn_forward(x, h0, W_hx, W_hh, b_h)
    print("=== 序列RNN前向传播结果 ===")
    print(f"输入形状: {x.shape}")
    print(f"隐藏状态序列形状: {h.shape}")
    print(f"第1个样本最后一步隐藏状态:\n{h[0, -1, :]}\n")

    # 构造随机上游梯度（模拟损失回传）
    dh = np.random.randn(*h.shape)
    
    # 反向传播
    dx, dh0, dW_hx, dW_hh, db_h = rnn_backward(dh, caches)
    print("=== 序列RNN反向传播梯度形状验证 ===")
    print(f"dx 形状: {dx.shape}  (匹配输入维度)")
    print(f"dh0形状: {dh0.shape} (匹配初始隐藏状态维度)")
    print(f"dW_hx形状: {dW_hx.shape} (匹配权重维度)")
    print(f"dW_hh形状: {dW_hh.shape} (匹配权重维度)")
    print(f"db_h 形状: {db_h.shape}  (匹配偏置维度)")


if __name__ == "__main__":
    # 1. 单步梯度校验
    gradient_check()
    # 2. 序列级RNN演示
    sequence_rnn_demo()

=== 单步RNN梯度验证结果（平均绝对误差） ===
dx_t   误差: 0.0000000000
dh_prev误差: 0.0000000000
dW_hx  误差: 0.0000000000
dW_hh  误差: 0.0000000000
db_h   误差: 0.0000000000

误差量级在 1e-9 ~ 1e-10，说明反向传播实现完全正确

=== 序列RNN前向传播结果 ===
输入形状: (2, 5, 3)
隐藏状态序列形状: (2, 5, 4)
第1个样本最后一步隐藏状态:
[-0.98120143  0.3444584   0.99191995 -0.82435358]

=== 序列RNN反向传播梯度形状验证 ===
dx 形状: (2, 5, 3)  (匹配输入维度)
dh0形状: (2, 4) (匹配初始隐藏状态维度)
dW_hx形状: (4, 3) (匹配权重维度)
dW_hh形状: (4, 4) (匹配权重维度)
db_h 形状: (4,)  (匹配偏置维度)


In [6]:
def count_deep_birnn_params(L, H, D, O):
    """
    计算深度双向RNN的总参数量（含所有RNN层+最终输出全连接层）
    参数:
        L: RNN总层数
        H: 单个方向的隐藏单元数
        D: 输入特征维度
        O: 输出维度
    返回:
        total_params: 模型总参数量
    """
    # ========== 1. 第1层双向RNN ==========
    # 输入维度为D，单向参数量 = H*D + H*H + H；双向×2
    layer1_params = 2 * (H * D + H * H + H)

    # ========== 2. 第2~L层双向RNN（共 L-1 层） ==========
    # 输入维度为上一层拼接后的 2H，单层双向参数量 = 2*(H*2H + H*H + H)
    if L > 1:
        single_mid_layer_params = 2 * (H * (2 * H) + H * H + H)
        middle_layers_params = (L - 1) * single_mid_layer_params
    else:
        middle_layers_params = 0

    # ========== 3. 最终输出全连接层 ==========
    # 输入维度为最后一层双向拼接输出 2H，输出维度O，含权重+偏置
    output_layer_params = O * (2 * H) + O

    # ========== 总参数量 ==========
    total_params = layer1_params + middle_layers_params + output_layer_params

    # 打印明细
    print("=" * 50)
    print(f"深度双向RNN配置：层数L={L}，单方向隐藏单元H={H}，输入维度D={D}，输出维度O={O}")
    print("-" * 50)
    print(f"第1层双向RNN参数量：{layer1_params}")
    if L > 1:
        print(f"第2~{L}层双向RNN（共{L-1}层）：{middle_layers_params}")
    print(f"输出全连接层参数量：{output_layer_params}")
    print("-" * 50)
    print(f"模型总参数量：{total_params}")
    print("=" * 50)

    return total_params


# ===================== 测试示例 =====================
if __name__ == "__main__":
    # 示例1：1层双向RNN
    print("【示例1：单层双向RNN】")
    count_deep_birnn_params(L=1, H=32, D=10, O=5)

    print("\n")

    # 示例2：3层深度双向RNN
    print("【示例2：3层深度双向RNN】")
    count_deep_birnn_params(L=3, H=64, D=20, O=10)

【示例1：单层双向RNN】
深度双向RNN配置：层数L=1，单方向隐藏单元H=32，输入维度D=10，输出维度O=5
--------------------------------------------------
第1层双向RNN参数量：2752
输出全连接层参数量：325
--------------------------------------------------
模型总参数量：3077


【示例2：3层深度双向RNN】
深度双向RNN配置：层数L=3，单方向隐藏单元H=64，输入维度D=20，输出维度O=10
--------------------------------------------------
第1层双向RNN参数量：10880
第2~3层双向RNN（共2层）：49408
输出全连接层参数量：1290
--------------------------------------------------
模型总参数量：61578


In [7]:
import torch
import torch.nn as nn


class BidirectionalRNNEncoder(nn.Module):
    """
    双向RNN编码器
    输入形状: (seq_len, batch, input_dim)
    输出1: 所有时间步拼接隐藏状态 (seq_len, batch, 2*hidden_dim)
    输出2: 最终序列表示 (batch, 2*hidden_dim)，由最后一层前向+后向最终状态拼接
    """
    def __init__(self, input_dim, hidden_dim, num_layers=1, dropout=0.0):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        # 原生双向RNN，batch_first=False 适配 (seq_len, batch, feature) 输入格式
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,    # 开启双向
            batch_first=False,
            dropout=dropout if num_layers > 1 else 0.0
        )

    def forward(self, x):
        """
        参数:
            x: 输入序列，形状 (seq_len, batch_size, input_dim)
        返回:
            all_hidden: 每个时间步的前向+后向拼接隐藏状态，形状 (seq_len, batch, 2*hidden_dim)
            final_hidden: 整个序列的最终编码表示，形状 (batch, 2*hidden_dim)
        """
        # 前向传播：output 为各时间步拼接结果，h_n 为所有层的最终隐藏状态
        all_hidden, h_n = self.rnn(x)

        # 提取最后一层的前向、后向最终隐藏状态并拼接
        # h_n 形状: (num_layers * 2, batch, hidden_dim)，顺序为 [层1前向, 层1后向, ..., 层L前向, 层L后向]
        last_forward = h_n[-2]    # 最后一层前向的最终状态 (batch, hidden_dim)
        last_backward = h_n[-1]   # 最后一层后向的最终状态 (batch, hidden_dim)
        final_hidden = torch.cat([last_forward, last_backward], dim=-1)  # (batch, 2*hidden_dim)

        return all_hidden, final_hidden


# ===================== 测试验证 =====================
if __name__ == "__main__":
    # 超参数设置
    seq_len = 10    # 序列长度
    batch = 4       # batch大小
    input_dim = 8   # 输入特征维度
    hidden_dim = 16 # 单方向隐藏单元数
    num_layers = 2  # RNN层数

    # 初始化模型
    encoder = BidirectionalRNNEncoder(
        input_dim=input_dim,
        hidden_dim=hidden_dim,
        num_layers=num_layers
    )

    # 构造随机输入，形状严格遵循 (seq_len, batch, input_dim)
    x = torch.randn(seq_len, batch, input_dim)

    # 前向传播
    all_hidden, final_hidden = encoder(x)

    # 打印形状验证
    print("=" * 50)
    print(f"输入序列形状: {tuple(x.shape)}")
    print(f"所有时间步拼接隐藏状态: {tuple(all_hidden.shape)}")
    print(f"最终序列表示形状: {tuple(final_hidden.shape)}")
    print("=" * 50)

输入序列形状: (10, 4, 8)
所有时间步拼接隐藏状态: (10, 4, 32)
最终序列表示形状: (4, 32)


In [8]:
import torch
import torch.nn as nn
import numpy as np


class SkipGramNegativeSampling(nn.Module):
    def __init__(self, vocab_size, embed_dim, word_counts, neg_k=5):
        """
        Skip-gram 负采样模型
        参数:
            vocab_size: 词汇表大小
            embed_dim: 词向量维度
            word_counts: 每个词的出现频次列表，索引对应词ID
            neg_k: 每个正样本采样的负样本数量
        """
        super().__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.neg_k = neg_k

        # 输入嵌入层：存储中心词向量 v_c
        self.in_embed = nn.Embedding(vocab_size, embed_dim)
        # 输出嵌入层：存储上下文/负样本词向量 u_o / u_k
        self.out_embed = nn.Embedding(vocab_size, embed_dim)

        # 均匀初始化词向量（Word2Vec标准初始化方式）
        nn.init.uniform_(self.in_embed.weight, -0.5/embed_dim, 0.5/embed_dim)
        nn.init.uniform_(self.out_embed.weight, -0.5/embed_dim, 0.5/embed_dim)

        # 构建负采样的噪声分布
        self._build_noise_distribution(word_counts)

    def _build_noise_distribution(self, word_counts):
        """构建一元语法3/4次方的噪声分布，用于负采样"""
        # 计算频次的0.75次方
        pow_counts = np.array(word_counts, dtype=np.float32) ** 0.75
        # 归一化为概率分布
        self.noise_probs = pow_counts / pow_counts.sum()
        self.noise_dist = torch.tensor(self.noise_probs, dtype=torch.float32)

    def get_negative_samples(self, batch_size):
        """
        批量采样负样本
        返回形状: (batch_size, neg_k)
        """
        neg_samples = torch.multinomial(
            self.noise_dist,
            num_samples=batch_size * self.neg_k,
            replacement=True
        ).view(batch_size, self.neg_k)
        return neg_samples

    def forward(self, center_words, context_words):
        """
        前向传播，计算负采样损失
        参数:
            center_words: 中心词ID，形状 (batch_size,)
            context_words: 真实上下文词ID，形状 (batch_size,)
        返回:
            loss: batch平均负对数似然损失（标量）
        """
        batch_size = center_words.shape[0]

        # 1. 取出中心词向量 v_c，形状 (batch_size, embed_dim)
        v_c = self.in_embed(center_words)

        # 2. 取出正样本上下文词向量 u_o，形状 (batch_size, embed_dim)
        u_o = self.out_embed(context_words)

        # 3. 采样负样本并取出对应向量 u_k，形状 (batch_size, neg_k, embed_dim)
        neg_samples = self.get_negative_samples(batch_size)
        u_k = self.out_embed(neg_samples)

        # 4. 正样本对数似然 log σ(u_o^T v_c)
        pos_score = torch.sum(v_c * u_o, dim=1)  # 逐元素相乘后求和，等价于内积
        pos_loss = torch.log(torch.sigmoid(pos_score))

        # 5. 负样本对数似然 Σ log σ(-u_k^T v_c)
        # 批量矩阵乘法: (batch, neg_k, dim) @ (batch, dim, 1) -> (batch, neg_k, 1)
        neg_score = torch.bmm(u_k, v_c.unsqueeze(2)).squeeze(2)
        neg_loss = torch.sum(torch.log(torch.sigmoid(-neg_score)), dim=1)

        # 6. 总损失 = 负对数似然，取batch平均
        total_loss = -torch.mean(pos_loss + neg_loss)

        return total_loss


# ===================== 测试运行 =====================
if __name__ == "__main__":
    # 模拟语料词频（索引对应词ID）
    word_counts = [120, 95, 72, 50, 38, 25, 18, 10]
    vocab_size = len(word_counts)
    embed_dim = 16
    neg_k = 5  # 每个正样本采5个负样本

    # 初始化模型
    model = SkipGramNegativeSampling(
        vocab_size=vocab_size,
        embed_dim=embed_dim,
        word_counts=word_counts,
        neg_k=neg_k
    )

    # 构造测试batch
    batch_size = 4
    center_words = torch.tensor([0, 2, 4, 6], dtype=torch.long)   # 中心词ID
    context_words = torch.tensor([1, 3, 5, 7], dtype=torch.long)  # 对应上下文词ID

    # 前向计算损失
    loss = model(center_words, context_words)

    print("=" * 50)
    print(f"词汇表大小: {vocab_size}")
    print(f"词向量维度: {embed_dim}")
    print(f"单样本负采样数: {neg_k}")
    print(f"Batch大小: {batch_size}")
    print(f"当前Batch平均损失: {loss.item():.4f}")
    print("=" * 50)

词汇表大小: 8
词向量维度: 16
单样本负采样数: 5
Batch大小: 4
当前Batch平均损失: 4.1591


In [9]:
import numpy as np


def cbow_forward_loss(context_indices, target_indices, W, W_out):
    """
    CBOW 前向传播 + 完整Softmax交叉熵损失计算
    参数:
        context_indices: 上下文词索引，形状 (batch_size, context_size)
        target_indices: 中心词（目标）索引，形状 (batch_size,)
        W: 输入权重矩阵，形状 (vocab_size, embed_dim)
        W_out: 输出权重矩阵，形状 (embed_dim, vocab_size)
    返回:
        loss: batch平均交叉熵损失（标量）
    """
    batch_size, context_size = context_indices.shape
    vocab_size, embed_dim = W.shape

    # ========== 1. 提取上下文词向量，求平均得到隐藏层 ==========
    # 从W中按索引取出所有上下文词向量，形状 (batch_size, context_size, embed_dim)
    context_vectors = W[context_indices]
    # 在上下文维度求平均，得到隐藏层 h，形状 (batch_size, embed_dim)
    h = np.mean(context_vectors, axis=1)

    # ========== 2. 计算全词汇表得分 logits ==========
    # h @ W_out 形状 (batch_size, vocab_size)，每行对应一个样本的全词表得分
    z = h @ W_out

    # ========== 3. 数值稳定的 Softmax 计算 ==========
    # 减去每行最大值，防止指数运算溢出
    z_shifted = z - np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(z_shifted)
    softmax_probs = exp_z / np.sum(exp_z, axis=1, keepdims=True)

    # ========== 4. 计算交叉熵损失（负对数似然） ==========
    # 取出每个样本对应目标词的概率
    batch_idx = np.arange(batch_size)
    target_probs = softmax_probs[batch_idx, target_indices]
    # 加极小值避免 log(0) 数值错误
    loss = -np.mean(np.log(target_probs + 1e-10))

    return loss


# ===================== 测试验证 =====================
if __name__ == "__main__":
    np.random.seed(42)

    # 超参数设置
    vocab_size = 10   # 词汇表大小 V
    embed_dim = 4     # 词向量维度 d
    context_size = 2  # 每个样本的上下文词数量
    batch_size = 3    # batch大小

    # 按题目指定形状初始化权重
    W = np.random.randn(vocab_size, embed_dim)       # 输入权重 (V, d)
    W_out = np.random.randn(embed_dim, vocab_size)   # 输出权重 (d, V)

    # 构造测试数据：上下文词索引 + 中心词索引
    context_indices = np.array([
        [0, 2],   # 第1个样本：上下文词0、2
        [1, 3],   # 第2个样本：上下文词1、3
        [4, 6]    # 第3个样本：上下文词4、6
    ])
    target_indices = np.array([1, 4, 5])  # 对应三个样本的中心词

    # 计算损失
    loss = cbow_forward_loss(context_indices, target_indices, W, W_out)

    print("=" * 50)
    print(f"词汇表大小: {vocab_size}, 词向量维度: {embed_dim}")
    print(f"上下文窗口大小: {context_size}, Batch大小: {batch_size}")
    print(f"输入权重形状: {W.shape}, 输出权重形状: {W_out.shape}")
    print(f"CBOW 平均交叉熵损失: {loss:.6f}")
    print("=" * 50)

词汇表大小: 10, 词向量维度: 4
上下文窗口大小: 2, Batch大小: 3
输入权重形状: (10, 4), 输出权重形状: (4, 10)
CBOW 平均交叉熵损失: 3.463793


In [10]:
import numpy as np


def softmax(x):
    """行级数值稳定Softmax，防止指数运算溢出"""
    # 减去每行最大值，保证数值稳定性
    x_shifted = x - np.max(x, axis=-1, keepdims=True)
    exp_x = np.exp(x_shifted)
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)


def scaled_dot_product_attention(Q, K, V, d_k, print_steps=True):
    """
    缩放点积注意力（无掩码）
    参数:
        Q: 查询矩阵，形状 (seq_q, d_k)
        K: 键矩阵，形状 (seq_k, d_k)
        V: 值矩阵，形状 (seq_k, d_v)
        d_k: 键/查询的维度
        print_steps: 是否打印中间步骤结果
    返回:
        output: 注意力输出矩阵，形状 (seq_q, d_v)
        attn_weights: 注意力权重矩阵，形状 (seq_q, seq_k)
    """
    # ========== 步骤1：计算得分矩阵 S = Q @ K^T ==========
    scores = np.dot(Q, K.T)
    if print_steps:
        print("=" * 60)
        print("步骤1：得分矩阵 S = QK^T")
        print(f"形状: {scores.shape}")
        print(scores)
        print()

    # ========== 步骤2：缩放得分 ==========
    scaled_scores = scores / np.sqrt(d_k)
    if print_steps:
        print(f"步骤2：缩放后得分 (除以√d_k，d_k={d_k})")
        print(f"形状: {scaled_scores.shape}")
        print(scaled_scores)
        print()

    # ========== 步骤3：Softmax 得到注意力权重 ==========
    attn_weights = softmax(scaled_scores)
    if print_steps:
        print("步骤3：Softmax 注意力权重矩阵")
        print(f"形状: {attn_weights.shape}")
        print(attn_weights)
        print(f"校验：每行和为 {np.sum(attn_weights, axis=1)}")
        print()

    # ========== 步骤4：加权求和得到输出 ==========
    output = np.dot(attn_weights, V)
    if print_steps:
        print("步骤4：注意力最终输出 Output = Attention(Q,K,V)")
        print(f"形状: {output.shape}")
        print(output)
        print("=" * 60)

    return output, attn_weights


# ===================== 题目数值测试 =====================
if __name__ == "__main__":
    # 按题目维度构造输入矩阵
    # Q ∈ R^(2×4)，K ∈ R^(3×4)，V ∈ R^(3×5)，d_k = 4
    Q = np.array([
        [1, 2, 3, 4],
        [2, 1, 4, 3]
    ])
    K = np.array([
        [1, 0, 1, 0],
        [0, 1, 0, 1],
        [1, 1, 0, 0]
    ])
    V = np.array([
        [1, 2, 3, 4, 5],
        [6, 7, 8, 9, 10],
        [11, 12, 13, 14, 15]
    ])
    d_k = 4

    # 执行注意力计算
    output, attn_weights = scaled_dot_product_attention(Q, K, V, d_k)

步骤1：得分矩阵 S = QK^T
形状: (2, 3)
[[4 6 3]
 [6 4 3]]

步骤2：缩放后得分 (除以√d_k，d_k=4)
形状: (2, 3)
[[2.  3.  1.5]
 [3.  2.  1.5]]

步骤3：Softmax 注意力权重矩阵
形状: (2, 3)
[[0.2312239  0.62853172 0.14024438]
 [0.62853172 0.2312239  0.14024438]]
校验：每行和为 [1. 1.]

步骤4：注意力最终输出 Output = Attention(Q,K,V)
形状: (2, 5)
[[5.54510243 6.54510243 7.54510243 8.54510243 9.54510243]
 [3.55856332 4.55856332 5.55856332 6.55856332 7.55856332]]


In [11]:
import numpy as np


def softmax(x):
    """数值稳定的Softmax，对最后一维做归一化，兼容任意批量维度"""
    x_max = np.max(x, axis=-1, keepdims=True)
    exp_x = np.exp(x - x_max)
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)


def scaled_dot_product_attention(Q, K, V, d_k):
    """
    通用缩放点积注意力，支持多头批量维度
    参数形状: Q/K/V = (..., seq_len, d_k/d_v)，...为任意前置维度（batch、头数）
    """
    # 1. 计算得分矩阵：最后两维做矩阵乘法
    scores = Q @ np.swapaxes(K, -2, -1)  # 形状 (..., seq_q, seq_k)
    # 2. 按 d_k 缩放
    scaled_scores = scores / np.sqrt(d_k)
    # 3. Softmax 得到注意力权重
    attn_weights = softmax(scaled_scores)
    # 4. 加权求和得到输出
    output = attn_weights @ V  # 形状 (..., seq_q, d_v)
    return output, attn_weights


class MultiHeadAttention:
    def __init__(self, d_model, num_heads):
        assert d_model % num_heads == 0, "d_model 必须能被 num_heads 整除"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # 每个头的查询/键维度
        self.d_v = self.d_k              # 题目要求 d_v = d_model/num_heads

        # 初始化4组线性投影权重（固定随机种子，结果可复现）
        np.random.seed(42)
        self.W_q = np.random.randn(d_model, d_model) * 0.1
        self.W_k = np.random.randn(d_model, d_model) * 0.1
        self.W_v = np.random.randn(d_model, d_model) * 0.1
        self.W_o = np.random.randn(d_model, d_model) * 0.1

    def forward(self, X):
        """
        多头注意力前向传播
        参数:
            X: 输入序列，形状 (seq_len, batch, d_model)
        返回:
            output: 输出序列，形状与输入完全相同 (seq_len, batch, d_model)
        """
        seq_len, batch, _ = X.shape

        # ========== 步骤1：线性投影得到 Q、K、V ==========
        Q = X @ self.W_q  # (seq_len, batch, d_model)
        K = X @ self.W_k  # (seq_len, batch, d_model)
        V = X @ self.W_v  # (seq_len, batch, d_model)

        # ========== 步骤2：拆分为 num_heads 个独立头 ==========
        # 先 reshape 拆分最后一维：(seq_len, batch, num_heads, d_k)
        Q = Q.reshape(seq_len, batch, self.num_heads, self.d_k)
        K = K.reshape(seq_len, batch, self.num_heads, self.d_k)
        V = V.reshape(seq_len, batch, self.num_heads, self.d_v)

        # 转置维度，将 batch 和 头数 放到前面，方便批量计算注意力
        # 转置后形状：(batch, num_heads, seq_len, d_k)
        Q = np.transpose(Q, axes=(1, 2, 0, 3))
        K = np.transpose(K, axes=(1, 2, 0, 3))
        V = np.transpose(V, axes=(1, 2, 0, 3))

        # ========== 步骤3：每个头独立计算缩放点积注意力 ==========
        attn_output, _ = scaled_dot_product_attention(Q, K, V, self.d_k)
        # 输出形状：(batch, num_heads, seq_len, d_v)

        # ========== 步骤4：拼接所有头的输出 ==========
        # 转置回 (seq_len, batch, num_heads, d_v)
        attn_output = np.transpose(attn_output, axes=(2, 0, 1, 3))
        # 合并最后两维，拼接恢复为 d_model 维度
        concat_output = attn_output.reshape(seq_len, batch, self.d_model)

        # ========== 步骤5：最终线性层投影 ==========
        output = concat_output @ self.W_o  # (seq_len, batch, d_model)

        return output


# ===================== 测试验证 =====================
if __name__ == "__main__":
    # 题目指定参数
    d_model = 4
    num_heads = 2

    # 构造测试输入：序列长度3，batch大小2，特征维度4
    seq_len = 3
    batch = 2
    X = np.random.randn(seq_len, batch, d_model)

    # 初始化多头注意力并前向传播
    mha = MultiHeadAttention(d_model, num_heads)
    output = mha.forward(X)

    print("=" * 60)
    print(f"输入形状 (seq_len, batch, d_model): {X.shape}")
    print(f"输出形状 (seq_len, batch, d_model): {output.shape}")
    print(f"输出与输入形状是否一致：{output.shape == X.shape}")
    print("-" * 60)
    print(f"单头维度 d_k = d_v = {mha.d_k}")
    print(f"头数 num_heads = {num_heads}，拼接后维度 = {num_heads} × {mha.d_k} = {d_model}")
    print("=" * 60)

输入形状 (seq_len, batch, d_model): (3, 2, 4)
输出形状 (seq_len, batch, d_model): (3, 2, 4)
输出与输入形状是否一致：True
------------------------------------------------------------
单头维度 d_k = d_v = 2
头数 num_heads = 2，拼接后维度 = 2 × 2 = 4
